# 03 — Periodic Phase Encoding

**역할**: chunk 첫 step phase `(cos φ₀, sin φ₀)`를 global condition으로 추가한 **Periodic Phase** 모델을 60 epoch 학습.

**산출물**: `checkpoints/phase_periodic_ckpt.pt`, `figures/phase_periodic_loss.png`

**예상 소요**: **~25분**

---

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `pcdp/configs.py`(`periodic_phase` config), `pcdp/experiment_runner.py`(data/model/scheduler/train-or-load wrapper), `pcdp/models.py`(`build_periodic_phase_dp_model`), `pcdp/training.py`(DDPM loop + `periodic_phase_cond_fn`), `pcdp/dataset.py`(`encode_phase_cossin`), `pcdp/experiment_plots.py`(loss curve)에 있습니다.

In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    project_root = Path.cwd()
    requirements_path = project_root / "requirements.txt"
    install_command = [sys.executable, "-m", "pip", "install"]

    if requirements_path.exists():
        # requirements.txt pins the runtime stack; -e . installs this repo from pyproject.toml.
        install_command.extend(["-r", str(requirements_path), "-e", str(project_root)])
    else:
        # Fallback to pyproject.toml dependencies if requirements.txt is unavailable.
        install_command.extend(["-e", str(project_root)])

    subprocess.check_call(install_command)

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config

In [ ]:
import torch

from pcdp.configs import get_experiment_config
from pcdp.experiment_plots import plot_loss_curve
from pcdp.experiment_runner import (
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    train_or_load_checkpoint,
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('periodic_phase')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 3. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 4. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 5. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 6. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)
